# Tables

Builds the tables (Tables 1-3) and the supplementary tables (S4 File, Tables A-E) from the result files. It runs no new analysis, so run notebooks 01-05 first.

| Table | Content | Reads |
|---|---|---|
| Table 1 | Characteristics of participants | `table1_characteristics.csv` |
| Table 2 | Main results for the 12 primary outcomes | `key_percentages.csv`, `primary_tests.csv` |
| Table 3 | Group differences after adjustment | `model_summary.csv`, `odds_ratios.csv` |
| Table A | All answers to every question | `item_distributions.csv` |
| Table B | Adjusted odds ratios for all predictors | `odds_ratios.csv`, `term_tests.csv` |
| Table C | Sensitivity analysis of answer coding | `primary_tests.csv`, `sensitivity_tests.csv`, `sensitivity_conclusions.csv` |
| Table D | Correlations between views | `correlations.csv` |
| Table E | Combinations of parties held accountable | `accountability_patterns.csv`, `accountability_number_test.csv` |

**Writes to `results/tables/`:** one CSV per table, `Tables_main.docx` and `Tables_supplementary.docx` (S4 File)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Mm, Pt

RESULTS = Path("../results")
TABLE_DIR = RESULTS / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

GROUPS = ["Patient", "Healthcare professional", "AI developer"]
COLS = {"Patient": "Patients", "Healthcare professional": "Healthcare professionals",
        "AI developer": "AI developers", "Total": "All"}


def R(name):
    return pd.read_csv(RESULTS / name, keep_default_na=False)


def fp(p):
    p = float(p)
    return "<0.001" if p < 0.001 else format(p, ".3f")


def f2(x):
    return format(float(x), ".2f")


key = R("key_percentages.csv")
N = {g: int(key.loc[key["group"] == g, "n"].iloc[0]) for g in GROUPS + ["Total"]}
HEAD = {g: COLS[g] + " (n = " + str(N[g]) + ")" for g in GROUPS + ["Total"]}
HEAD

{'Patient': 'Patients (n = 48)',
 'Healthcare professional': 'Healthcare professionals (n = 58)',
 'AI developer': 'AI developers (n = 147)',
 'Total': 'All (n = 253)'}

## Table 1: characteristics of participants

In [2]:
t1 = R("table1_characteristics.csv")
t1 = t1[t1["characteristic"] != "Participants, n"]
table1 = pd.DataFrame({"Characteristic": np.where(t1["category"] == "", t1["characteristic"], "  " + t1["category"]),
                       **{HEAD[g]: t1[g] for g in GROUPS + ["Total"]}})
table1.to_csv(TABLE_DIR / "table_1_characteristics.csv", index=False)
T1_TITLE = "Table 1 Characteristics of participants by group"
T1_NOTE = ("Values are n (%). Percentages use the column total, except experience, which uses healthcare "
           "professionals and AI developers only (n = " + str(N["Healthcare professional"] + N["AI developer"]) + "); "
           "patients were not asked about experience. All answer options on the form are shown, including those no one chose. "
           "No significance tests were run on participant characteristics.")
table1

,Characteristic,Patients (n = 48),Healthcare professionals (n = 58),AI developers (n = 147),All (n = 253)
1,Gender,,,,
2,Male,25 (52.1),26 (44.8),95 (64.6),146 (57.7)
3,Female,23 (47.9),32 (55.2),52 (35.4),107 (42.3)
4,Non-binary,0 (0.0),0 (0.0),0 (0.0),0 (0.0)
5,Prefer not to say,0 (0.0),0 (0.0),0 (0.0),0 (0.0)
6,Age group (years),,,,
7,18-24,11 (22.9),21 (36.2),78 (53.1),110 (43.5)
8,25-34,16 (33.3),19 (32.8),41 (27.9),76 (30.0)
9,35-44,18 (37.5),14 (24.1),25 (17.0),57 (22.5)
10,45-54,3 (6.2),4 (6.9),2 (1.4),9 (3.6)


## Table 2: main results for the 12 primary outcomes

In [3]:
prim = R("primary_tests.csv")
SHOW = [  # (outcome number, row label, measure in key_percentages.csv, note marker)
    (1,  "Q7 Concerned about AI perpetuating bias (any concern)", "Q7 bias: any concern (Somewhat or more)", "a"),
    (2,  "Q8 Agree AI protects patient data and privacy", "Q8 privacy is protected: agree or strongly agree", "b"),
    (3,  "Q9 Clear explanations are important (any importance)", "Q9 explanations: any importance (Somewhat or more)", "a"),
    (4,  "Q10 Patients should always be informed: Yes", "Q10 patients should always be informed: Yes", "c"),
    (5,  "Q11 Accountable: AI developers", "Q11 accountable: AI developers", "d"),
    (6,  "Q11 Accountable: healthcare professional", "Q11 accountable: healthcare professional", "d"),
    (7,  "Q11 Accountable: hospital or clinic", "Q11 accountable: hospital or clinic", "d"),
    (8,  "Q12 Complete trust in AI compared with professionals", "Q12 trust: complete trust", "b"),
    (9,  "Q13 Agree AI could undermine autonomy", "Q13 autonomy could be undermined: agree or strongly agree", "b"),
    (10, "Q14 Confident current ethical guidelines are sufficient (any confidence)", "Q14 guidelines are enough: any confidence (Somewhat or more)", "a"),
    (11, "Q15 Not comfortable with AI making life-and-death decisions without a human", "Q15 comfortable with no human: No", "c"),
    (12, "Q16 Concerned about AI increasing disparities (any concern)", "Q16 disparities: any concern (Somewhat or more)", "a"),
]
rows = []
for num, label, measure, mark in SHOW:
    p = prim[prim["outcome_no"].astype(int) == num].iloc[0]
    row = {"Outcome": label + " [" + mark + "]"}
    for g in GROUPS + ["Total"]:
        k = key[(key["measure"] == measure) & (key["group"] == g)].iloc[0]
        row[HEAD[g]] = format(float(k["pct"]), ".1f") + " (" + format(float(k["ci_low"]), ".1f") + " to " + format(float(k["ci_high"]), ".1f") + ")"
    stat = ("chi2 = " if p["test"] == "Chi-square" else "H = ") + f2(p["statistic"]) + ", df " + str(int(float(p["df"])))
    es = ("V = " if p["effect_size_name"] == "Cramer's V" else "eps2 = ") + f2(p["effect_size"]) + " (" + f2(p["es_ci_low"]) + " to " + f2(p["es_ci_high"]) + ")"
    row.update({"Test": stat, "p": fp(p["p"]), "Holm p": fp(p["p_holm"]), "Effect size (95% CI)": es})
    rows.append(row)
table2 = pd.DataFrame(rows)
table2.to_csv(TABLE_DIR / "table_2_main_results.csv", index=False)
T2_TITLE = "Table 2 Views on ethical issues of AI in healthcare, by group"
T2_NOTE = ("Values are % (95% confidence interval, Wilson method). Group comparison: chi-square test (binary and Yes/No/Not sure "
           "outcomes) or Kruskal-Wallis test (ordered outcomes) across the three groups; Holm correction across the 12 primary tests. "
           "Effect size: Cramer's V (V) or epsilon-squared (eps2), with 95% bootstrap intervals; these measures cannot be below zero, "
           "so a lower limit above zero does not by itself indicate a difference. "
           "[a] The test used the full 1-5 scale (Not = 1, Neutral = 2, Somewhat = 3, then 4 and 5); "
           "the percentage shows answers of Somewhat or higher. "
           "[b] The test used the full answer scale; the percentage shows one part of it. "
           "[c] The test used all three answers (Yes, No, Not sure). "
           "[d] Participants could choose more than one party.")
table2

,Outcome,Patients (n = 48),Healthcare professionals (n = 58),AI developers (n = 147),All (n = 253),Test,p,Holm p,Effect size (95% CI)
0,Q7 Concerned about AI perpetuating bias (any c...,54.2 (40.3 to 67.4),50.0 (37.5 to 62.5),53.7 (45.7 to 61.6),53.0 (46.8 to 59.0),"H = 0.34, df 2",0.843,1.000,eps2 = 0.00 (0.00 to 0.04)
1,Q8 Agree AI protects patient data and privacy [b],37.5 (25.2 to 51.6),43.1 (31.2 to 55.9),39.5 (31.9 to 47.5),39.9 (34.1 to 46.1),"H = 0.83, df 2",0.662,1.000,eps2 = 0.00 (0.00 to 0.04)
2,Q9 Clear explanations are important (any impor...,79.2 (65.7 to 88.3),75.9 (63.5 to 85.0),76.9 (69.4 to 83.0),77.1 (71.5 to 81.8),"H = 5.92, df 2",0.052,0.570,eps2 = 0.02 (0.00 to 0.08)
3,Q10 Patients should always be informed: Yes [c],75.0 (61.2 to 85.1),65.5 (52.7 to 76.4),78.2 (70.9 to 84.1),74.7 (69.0 to 79.7),"chi2 = 3.83, df 4",0.429,1.000,V = 0.09 (0.05 to 0.20)
4,Q11 Accountable: AI developers [d],58.3 (44.3 to 71.2),50.0 (37.5 to 62.5),70.7 (62.9 to 77.5),63.6 (57.5 to 69.3),"chi2 = 8.46, df 2",0.015,0.175,V = 0.18 (0.07 to 0.32)
5,Q11 Accountable: healthcare professional [d],66.7 (52.5 to 78.3),63.8 (50.9 to 74.9),70.1 (62.2 to 76.9),68.0 (62.0 to 73.4),"chi2 = 0.80, df 2",0.670,1.000,V = 0.06 (0.02 to 0.20)
6,Q11 Accountable: hospital or clinic [d],66.7 (52.5 to 78.3),48.3 (35.9 to 60.8),53.1 (45.0 to 60.9),54.5 (48.4 to 60.6),"chi2 = 3.89, df 2",0.143,1.000,V = 0.12 (0.03 to 0.26)
7,Q12 Complete trust in AI compared with profess...,10.4 (4.5 to 22.2),10.3 (4.8 to 20.8),11.6 (7.3 to 17.7),11.1 (7.8 to 15.5),"H = 0.04, df 2",0.981,1.000,eps2 = 0.00 (0.00 to 0.03)
8,Q13 Agree AI could undermine autonomy [b],35.4 (23.4 to 49.6),39.7 (28.1 to 52.5),34.7 (27.5 to 42.7),36.0 (30.3 to 42.1),"H = 0.07, df 2",0.965,1.000,eps2 = 0.00 (0.00 to 0.03)
9,Q14 Confident current ethical guidelines are s...,35.4 (23.4 to 49.6),48.3 (35.9 to 60.8),43.5 (35.8 to 51.6),43.1 (37.1 to 49.2),"H = 2.10, df 2",0.351,1.000,eps2 = 0.01 (0.00 to 0.05)


## Table 3: group differences after adjustment

In [4]:
summ = R("model_summary.csv")
odds = R("odds_ratios.csv")
T3_LABEL = {num: label for num, label, _, _ in SHOW}
T3_LABEL[8] = "Q12 Trust in AI compared with professionals (full 3-point scale)"
T3_LABEL[9] = "Q13 Agree AI could undermine autonomy (full 5-point scale)"
T3_LABEL[12] = "Q16 Concern about AI increasing disparities (full 5-point scale)"


def or_text(o, pred):
    r = o.loc[pred]
    return f2(r["or"]) + " (" + f2(r["ci_low"]) + " to " + f2(r["ci_high"]) + ")"


rows = []
for _, s in summ.iterrows():
    num = int(s["outcome_no"])
    o = odds[odds["outcome_no"].astype(int) == num].set_index("predictor")
    rows.append({
        "Outcome": T3_LABEL[num] + (" [less stable]" if s["stability"] == "less stable" else ""),
        "Model": s["model"],
        "Healthcare professionals vs patients, OR (95% CI)": or_text(o, "Healthcare professional (vs patient)"),
        "AI developers vs patients, OR (95% CI)": or_text(o, "AI developer (vs patient)"),
        "Group effect, LR chi2 (2 df)": f2(s["group_lr_chi2"]),
        "p": fp(s["group_p"]),
        "Holm p": fp(s["group_p_holm"]),
    })
table3 = pd.DataFrame(rows)
table3.to_csv(TABLE_DIR / "table_3_adjusted_results.csv", index=False)
T3_TITLE = "Table 3 Group differences after adjustment for age, gender, continent and familiarity"
T3_NOTE = ("OR, odds ratio; CI, confidence interval; LR, likelihood ratio. Logistic regression for binary outcomes; ordinal logistic "
           "regression (proportional odds) for trust, autonomy and disparities. Q8 (privacy), Q7 (bias), Q9 (explanations) and Q14 "
           "(guidelines) failed the proportional odds check and are modelled as agree or strongly agree (Q8) or Somewhat or higher "
           "(Q7, Q9, Q14) versus the rest, as set out before fitting. Each model includes group, age group "
           "(18-24, 25-34, 35 and over), gender, continent (Asia or other) and familiarity with AI (1-5). OR above 1: higher odds of "
           "the outcome (or of a higher answer) than patients. Holm correction across the 12 group tests. "
           "[less stable] fewer than 70 participants in the smaller outcome group (fewer than 10 per model parameter); read with care.")
table3

,Outcome,Model,"Healthcare professionals vs patients, OR (95% CI)","AI developers vs patients, OR (95% CI)","Group effect, LR chi2 (2 df)",p,Holm p
0,Q7 Concerned about AI perpetuating bias (any c...,logistic,0.88 (0.40 to 1.96),1.14 (0.56 to 2.31),0.61,0.739,1.000
1,Q8 Agree AI protects patient data and privacy,logistic,1.25 (0.56 to 2.82),0.94 (0.45 to 1.93),0.75,0.687,1.000
2,Q9 Clear explanations are important (any impor...,logistic,0.97 (0.35 to 2.64),1.42 (0.58 to 3.49),1.13,0.567,1.000
3,Q10 Patients should always be informed: Yes [l...,logistic,0.92 (0.37 to 2.29),1.82 (0.78 to 4.26),4.01,0.135,1.000
4,Q11 Accountable: AI developers,logistic,0.66 (0.30 to 1.47),1.71 (0.83 to 3.54),8.30,0.016,0.189
5,Q11 Accountable: healthcare professional,logistic,0.96 (0.41 to 2.22),1.22 (0.57 to 2.58),0.55,0.759,1.000
6,Q11 Accountable: hospital or clinic,logistic,0.56 (0.24 to 1.29),0.66 (0.31 to 1.38),1.98,0.371,1.000
7,Q12 Trust in AI compared with professionals (f...,ordinal logistic,0.67 (0.27 to 1.68),0.88 (0.39 to 2.02),0.84,0.658,1.000
8,Q13 Agree AI could undermine autonomy (full 5-...,ordinal logistic,1.23 (0.58 to 2.58),1.50 (0.78 to 2.89),1.54,0.462,1.000
9,Q14 Confident current ethical guidelines are s...,logistic,1.84 (0.82 to 4.17),1.61 (0.78 to 3.34),2.41,0.300,1.000


## Supplementary tables A-E (S4 File)

In [5]:
# Table A: every answer to every question
dist = R("item_distributions.csv")
QLAB = {"familiarity": "Q6 Familiarity with AI in healthcare", "bias_concern": "Q7 Concern: AI perpetuating biases",
        "privacy_protection": "Q8 AI adequately protects patient data and privacy",
        "explanation_importance": "Q9 Importance of clear explanations", "inform_patients": "Q10 Patients should always be informed",
        "trust": "Q12 Trust in AI compared with professionals", "autonomy_undermined": "Q13 AI could undermine autonomy",
        "guidelines_confidence": "Q14 Confidence that ethical guidelines are sufficient",
        "no_human_life_death": "Q15 Comfortable with AI life-and-death decisions without a human",
        "disparities_concern": "Q16 Concern: AI increasing disparities in access"}
rows = []
for var, qlabel in QLAB.items():
    rows.append({"Question and answer": qlabel, **{HEAD[g]: "" for g in GROUPS + ["Total"]}})
    for _, d in dist[dist["variable"] == var].iterrows():
        rows.append({"Question and answer": "  " + d["answer"],
                     **{HEAD[g]: str(int(d[g + "_n"])) + " (" + format(float(d[g + "_pct"]), ".1f") + ")" for g in GROUPS + ["Total"]}})
tA = pd.DataFrame(rows)
A = ("Table A All answers to every opinion question, by group",
     "Values are n (%). Answers are listed in the order used in the analysis. For Q7, Q9, Q14 and Q16 this is "
     "Not < Neutral < Somewhat < ... < Extremely; on the form, Neutral was the third option. Q11 (accountability) is in Table 2 and Table E.")

# Table B: all adjusted odds ratios
terms = R("term_tests.csv")
o = odds.copy()
o["text"] = o["or"].map(f2) + " (" + o["ci_low"].map(f2) + " to " + o["ci_high"].map(f2) + ")"
TERM_OF = {"Healthcare professional (vs patient)": "group", "AI developer (vs patient)": "group",
           "Age 25-34 (vs 18-24)": "age", "Age 35 and over (vs 18-24)": "age", "Female (vs male)": "gender",
           "Not Asia (vs Asia)": "continent", "Familiarity (per step, 1-5)": "familiarity"}
rows = []
for out in dict.fromkeys(o["outcome"]):
    num = int(o.loc[o["outcome"] == out, "outcome_no"].iloc[0])
    rows.append({"Outcome / predictor": out, "Adjusted OR (95% CI)": "", "Term LR test p": "", "Term BH p": ""})
    tt = terms[terms["outcome_no"].astype(int) == num].set_index("term")
    for pred in dict.fromkeys(o["predictor"]):
        r = o[(o["outcome"] == out) & (o["predictor"] == pred)].iloc[0]
        t = tt.loc[TERM_OF[pred]]
        rows.append({"Outcome / predictor": "  " + pred, "Adjusted OR (95% CI)": r["text"], "Term LR test p": fp(t["p"]),
                     "Term BH p": fp(t["p_bh_48"]) if str(t["p_bh_48"]) != "" else "(Holm " + fp(t["p_holm"]) + ")"})
tB = pd.DataFrame(rows)
B = ("Table B Adjusted odds ratios for all predictors in the 12 regression models",
     "OR, odds ratio; LR, likelihood ratio; BH, Benjamini-Hochberg. Reference categories: patient, age 18-24, male, Asia. "
     "Term p values test each characteristic as a whole (group and age have 2 df). Group terms are corrected by Holm across 12 tests "
     "(shown in brackets); other terms by Benjamini-Hochberg across 48 tests, added for reading only (not in the analysis plan).")

# Table C: sensitivity analysis
chk = R("sensitivity_conclusions.csv")
sens = R("sensitivity_tests.csv")
prim_idx = prim.set_index(prim["outcome_no"].astype(str))
rows = []
for _, c in chk[chk["outcome_no"] != "all 12"].iterrows():
    if c["version"].startswith("Main"):
        s = prim_idx.loc[str(c["outcome_no"])]
    else:
        s = sens[(sens["version"] == c["version"]) & (sens["outcome_no"].astype(str) == str(c["outcome_no"]))].iloc[0]
    es = ("V = " if s["effect_size_name"] == "Cramer's V" else "eps2 = ") + f2(s["effect_size"])
    rows.append({"Coding": c["version"], "Outcome": c["outcome"], "p": fp(c["p"]),
                 "Holm p (family of 12)": fp(c["p_holm_in_family_of_12"]), "Effect size": es})
for v in chk["version"].unique():
    n_diff = chk[(chk["version"] == v) & (chk["outcome_no"] == "all 12")]["difference_after_holm"].iloc[0]
    rows.append({"Coding": v, "Outcome": "Outcomes with a group difference after Holm (of 12)", "p": "",
                 "Holm p (family of 12)": "", "Effect size": str(n_diff)})
tC = pd.DataFrame(rows)
C = ("Table C Sensitivity analysis of answer coding: other codings of the four items with Neutral placed third",
     "Main: answer codes 1-5 (Not < Neutral < Somewhat < ... < Extremely), Kruskal-Wallis. Top two answers: top two answers "
     "versus the rest, chi-square. Form order: answer codes 1-5 in form order, Kruskal-Wallis. Somewhat or higher: Somewhat or more versus the rest, "
     "chi-square. In each version the four results replace the main ones in the family "
     "of 12 primary tests and the Holm correction is redone.")

# Table D: correlations
cr = R("correlations.csv")
tD = pd.DataFrame({"View 1": cr["var_1"], "View 2": cr["var_2"],
                   "Spearman rho (95% CI)": cr["rho"].map(f2) + " (" + cr["ci_low"].map(f2) + " to " + cr["ci_high"].map(f2) + ")",
                   "p": cr["p"].map(fp), "BH p": cr["p_bh"].map(fp)})
D = ("Table D Correlations between views on different ethical issues (exploratory)",
     "Spearman correlations, n = " + str(N["Total"]) + ". 95% CI by Fisher z with the Bonett-Wright standard error. BH, Benjamini-Hochberg "
     "correction across 28 pairs. Q7, Q9, Q14 and Q16 coded 1-5 (Not = 1, Neutral = 2, Somewhat = 3, then 4 and 5). "
     "These are associations at one time point and do not show cause.")

# Table E: accountability patterns
pat = R("accountability_patterns.csv")
test = R("accountability_number_test.csv").iloc[0]
tE = pd.DataFrame({"Pattern": np.where(pat["pattern_type"] == "number chosen", "Number of parties chosen: " + pat["pattern"].astype(str),
                                       "Combination: " + pat["pattern"].astype(str)),
                   **{HEAD[g]: pat[g] for g in GROUPS + ["Total"]}})
E = ("Table E How participants combined their answers on accountability (Q11)",
     "Values are n (%). Number of parties chosen by group (exploratory): chi-square = " + f2(test["statistic"]) + ", df "
     + str(int(float(test["df"]))) + ", p = " + fp(test["p"]) + ", Cramer's V = " + f2(test["cramers_v"]) + " (95% CI " + f2(test["ci_low"])
     + " to " + f2(test["ci_high"]) + ").")

for name, t in [("A_all_answers", tA), ("B_full_odds_ratios", tB), ("C_sensitivity", tC),
                ("D_correlations", tD), ("E_accountability_patterns", tE)]:
    t.to_csv(TABLE_DIR / ("table_" + name + ".csv"), index=False)
    print("Table", name, t.shape)

Table A_all_answers (54, 5)
Table B_full_odds_ratios (96, 4)
Table C_sensitivity (20, 5)
Table D_correlations (28, 5)
Table E_accountability_patterns (10, 5)


## Word files

Plain tables: Times New Roman 9 pt, a rule above and below the header and at the bottom, no shading. Wide tables go on landscape pages.

In [6]:
def cell_borders(cell, **edges):
    tcPr = cell._tc.get_or_add_tcPr()
    for old in tcPr.findall(qn("w:tcBorders")):
        tcPr.remove(old)
    b = OxmlElement("w:tcBorders")
    for edge in ("top", "left", "bottom", "right"):
        el = OxmlElement("w:" + edge)
        if edges.get(edge):
            el.set(qn("w:val"), "single"); el.set(qn("w:sz"), str(edges[edge])); el.set(qn("w:color"), "000000")
        else:
            el.set(qn("w:val"), "nil")
        b.append(el)
    tcPr.append(b)


def add_table(doc, frame, title, note, landscape=False):
    sec = doc.add_section() if len(doc.paragraphs) else doc.sections[0]
    sec.orientation = WD_ORIENT.LANDSCAPE if landscape else WD_ORIENT.PORTRAIT
    sec.page_width, sec.page_height = (Mm(297), Mm(210)) if landscape else (Mm(210), Mm(297))
    for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
        setattr(sec, m, Mm(18))
    doc.add_paragraph().add_run(title).bold = True
    t = doc.add_table(rows=1, cols=len(frame.columns))
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, c in enumerate(frame.columns):
        t.rows[0].cells[i].text = str(c)
        for run in t.rows[0].cells[i].paragraphs[0].runs:
            run.bold = True
    for _, row in frame.iterrows():
        cells = t.add_row().cells
        for i, v in enumerate(row):
            cells[i].text = str(v)
    last = len(t.rows) - 1
    for i, row in enumerate(t.rows):
        for cell in row.cells:
            cell_borders(cell, top=8 if i == 0 else None, bottom=6 if i == 0 else (8 if i == last else None))
    doc.add_paragraph(note)


def new_doc():
    d = Document()
    st = d.styles["Normal"]; st.font.name = "Times New Roman"; st.font.size = Pt(9)
    st.paragraph_format.space_after = Pt(2); st.paragraph_format.line_spacing = 1.0
    return d


main_doc = new_doc()
add_table(main_doc, table1, T1_TITLE, T1_NOTE)
add_table(main_doc, table2, T2_TITLE, T2_NOTE, landscape=True)
add_table(main_doc, table3, T3_TITLE, T3_NOTE, landscape=True)
main_doc.save(TABLE_DIR / "Tables_main.docx")

supp = new_doc()
for frame, (title, note), land in [(tA, A, False), (tB, B, False), (tC, C, False), (tD, D, False), (tE, E, True)]:
    add_table(supp, frame, title, note, landscape=land)
supp.save(TABLE_DIR / "Tables_supplementary.docx")
print("Saved Tables_main.docx and Tables_supplementary.docx")

Saved Tables_main.docx and Tables_supplementary.docx
